# SkillOpt demo — the on-disk contrast to GEPA (log-EDA)

**US3 of the agentbook thesis.** SkillOpt evolves the **skill-document** slice and is a
**batch trainer with on-disk state** — the opposite shape from GEPA's kernel-resident loop.
Per the thesis, the notebook's honest value here is *not* running the loop (that's
`scripts/train.py`'s job) but **post-hoc EDA of the on-disk trajectory log** a single run
produces — "use logs, not reruns." We read a real run at `/tmp/abook_skillopt_run`
(2 train / 4 val / 4 test SQuAD items, `claude_chat`→Bedrock) through the
`SkillOptOptimizer` adapter, map it onto the same `Session` entities GEPA used, and
compare the two optimizers.

## How to read this demo (step by step)

1. **Point at the on-disk run** — SkillOpt is a *batch trainer* whose state lives on disk (the opposite of GEPA's kernel-resident frontier). This demo does **log-EDA**: it reads a real captured run directory, it does not launch a new training run.
2. **Load the run artifacts** — `summary.json`, `history.json`, and per-item `predictions/.../results.jsonl` (the real eval outputs).
3. **Map the on-disk run onto the SAME `Session` entities** GEPA used — proving the substrate is state-agnostic (SC-003): one `Optimizer` contract hosts both a kernel-resident and an on-disk optimizer with no substrate-specific code.
4. **Contrast GEPA vs SkillOpt** — same loop contract, different slice (skill document vs. system prompt) and different state shape (on-disk vs. in-memory).

In [ ]:
from pathlib import Path

import pandas as pd
from utils import bootstrap

REPO = bootstrap()  # find repo root, load .env, put src/ on sys.path (no hardcoded path)

from agentbook.adapters.skillopt_adapter import SkillOptOptimizer
from agentbook.session import Session

RUN = "/tmp/abook_skillopt_run"  # a real claude_chat->Bedrock run (on-disk state)
print("run dir:", RUN)
print("on-disk artifacts:", sorted(p.name for p in Path(RUN).iterdir()))

In [2]:
import json

# 1) Per-item results across every results.jsonl (real rollout outcomes)
res = pd.DataFrame(SkillOptOptimizer.load_results(RUN))
print("=== per-item results (real SQuAD rollouts) ===")
display(res[["phase", "id", "predicted_answer", "gold_answers", "hard", "soft", "agent_ok"]])

# 2) Logged trajectories — the exact conversation.json files SkillOpt's reflect step parses
traj = pd.DataFrame(SkillOptOptimizer.load_trajectories(RUN))
print(f"\n=== {len(traj)} logged trajectories (reflect reads these from disk) ===")
display(traj)

# 3) Where the step actually spent time + tokens (history.json)
hist = json.load(open(f"{RUN}/history.json"))
step = hist[0]
print("\n=== step 1 breakdown (history.json) ===")
print("timing (s):", step["timing"])
print("tokens    :", step["tokens"])
print(
    "action    :",
    step["action"],
    "| rollout_hard:",
    step["rollout_hard"],
    "| best_score:",
    step["best_score"],
)

=== per-item results (real SQuAD rollouts) ===


,phase,id,predicted_answer,gold_answers,hard,soft,agent_ok
0,selection_eval_baseline,squad-val-9,American Football Conference,[American Football Conference],1,1.0,True
1,selection_eval_baseline,squad-val-8,golden anniversary,"[""golden anniversary"", gold-themed, gold]",1,1.0,True
2,steps,squad-val-0,Denver Broncos,[Denver Broncos],1,1.0,True
3,steps,squad-val-1,Carolina Panthers,[Carolina Panthers],1,1.0,True
4,steps,squad-val-9,American Football Conference,[American Football Conference],1,1.0,True
5,steps,squad-val-8,golden anniversary,"[""golden anniversary"", gold-themed, gold]",1,1.0,True
6,test_eval,squad-val-12,Levi's Stadium,"[Levi's Stadium, Levi's Stadium in the San Fra...",1,1.0,True
7,test_eval,squad-val-13,Santa Clara,[Santa Clara],1,1.0,True
8,test_eval_baseline,squad-val-12,Levi's Stadium,"[Levi's Stadium, Levi's Stadium in the San Fra...",1,1.0,True
9,test_eval_baseline,squad-val-13,"Santa Clara, California",[Santa Clara],0,0.8,True



=== 10 logged trajectories (reflect reads these from disk) ===


,phase,item_id,n_turns,roles
0,selection_eval_baseline,squad-val-8,2,"[None, system]"
1,selection_eval_baseline,squad-val-9,2,"[None, system]"
2,steps,squad-val-0,2,"[None, system]"
3,steps,squad-val-1,2,"[None, system]"
4,steps,squad-val-8,2,"[None, system]"
5,steps,squad-val-9,2,"[None, system]"
6,test_eval,squad-val-12,2,"[None, system]"
7,test_eval,squad-val-13,2,"[None, system]"
8,test_eval_baseline,squad-val-12,2,"[None, system]"
9,test_eval_baseline,squad-val-13,2,"[None, system]"



=== step 1 breakdown (history.json) ===
timing (s): {'rollout_s': 4.1, 'reflect_s': 24.9, 'aggregate_s': 0.0, 'select_s': 0.0, 'update_s': 0.0, 'evaluate_s': 3.5}
tokens    : {'analyst': {'calls': 1, 'prompt_tokens': 3, 'completion_tokens': 1142}, 'rollout': {'calls': 4, 'prompt_tokens': 12
, 'completion_tokens': 67}}
action    : reject | rollout_hard: 1.0 | best_score: 1.0


In [3]:
# Map the on-disk run onto the SAME Session entities GEPA used (no LLM — pure disk mapping)
seed_skill = Path(f"{RUN}/skills/skill_v0000.md").read_text()
session = Session(
    eval_set=["searchqa"],
    model_client=lambda *_a, **_k: None,
    slice_kind="skill_document",
    seed_artifact=seed_skill,
)

opt = SkillOptOptimizer(session, skillopt_root="/Users/mhuang/Documents/GitHub/SkillOpt-src")
summary = opt.sync_from_disk(RUN)

print("slice under evolution :", session.slice_kind)
print("candidates in session :", len(session.candidates), "(seed + best)")
print(
    "iterations recorded   :",
    len(session.iterations),
    "| actions:",
    [i.new_candidate_id is not None for i in session.iterations],
)
print("test_hard (best)      :", session.best_candidate().scores)
print(
    "steps/accepts/rejects :",
    summary["total_steps"],
    summary["total_accepts"],
    summary["total_rejects"],
)
print(
    "baseline->best test   :",
    summary["baseline_test_hard"],
    "->",
    summary["test_hard"],
    "(delta",
    summary["test_delta_hard"],
    ")",
)

slice under evolution : skill_document
candidates in session : 2 (seed + best)
iterations recorded   : 1 | actions: [False]
test_hard (best)      : {'test_hard': 1.0}
steps/accepts/rejects : 1 0 1
baseline->best test   : 0.5 -> 1.0 (delta 0.5 )


In [4]:
# GEPA (US1) vs SkillOpt (US3) — both ran live on Bedrock through the same substrate.
comparison = pd.DataFrame(
    [
        ["harness slice", "system prompt", "skill document (.md)"],
        [
            "optimizer state",
            "kernel-resident (in-memory)",
            "on-disk (predictions/, history.json, skills/)",
        ],
        ["LLM path to Bedrock", "litellm (in-process)", "claude CLI (subprocess)"],
        ["engine", "gepa.optimize()", "ReflACTTrainer.train()"],
        ["reflect input", "in-memory traces", "conversation.json parsed from disk"],
        ["real run", "168.8s, 34 metric calls, 4 cand", "39s, 11 calls, 1 step"],
        [
            "outcome (honest)",
            "best=seed (AIME too hard)",
            "edit gate-rejected; best=seed",
        ],
        [
            "substrate ops",
            "live kernel / MCP cells",
            "live kernel / MCP cells (identical)",
        ],
        [
            "notebook-fit",
            "HIGH (warm loop = the value)",
            "MEDIUM (value = log EDA, not running)",
        ],
    ],
    columns=["dimension", "GEPA (US1)", "SkillOpt (US3)"],
)
display(comparison)
print(
    "\nSC-003: one substrate hosted two optimizers, two harness slices, two state shapes — confirmed with real runs of both."
)

,dimension,GEPA (US1),SkillOpt (US3)
0,harness slice,system prompt,skill document (.md)
1,optimizer state,kernel-resident (in-memory),"on-disk (predictions/, history.json, skills/)"
2,LLM path to Bedrock,litellm (in-process),claude CLI (subprocess)
3,engine,gepa.optimize(),ReflACTTrainer.train()
4,reflect input,in-memory traces,conversation.json parsed from disk
5,real run,"168.8s, 34 metric calls, 4 cand","39s, 11 calls, 1 step"
6,outcome (honest),best=seed (AIME too hard),edit gate-rejected; best=seed
7,substrate ops,live kernel / MCP cells,live kernel / MCP cells (identical)
8,notebook-fit,HIGH (warm loop = the value),"MEDIUM (value = log EDA, not running)"



SC-003: one substrate hosted two optimizers, two harness slices, two state shapes — confirmed with real runs of both.


## Analysis — what is (and isn't) happening

- **The slice under evolution is the skill document.** The target `system_prompt` = a fixed QA role **plus** an injected *skill* section that starts empty (`"No learned rules yet. Rules will be added through the reflection process."`); SkillOpt's reflection appends rules to it across training.
- **Still single-turn Q&A — no tool calls.** The captured `results.jsonl` shows `n_turns: 1` per item and `response: "<answer>…</answer>"`; `conversation.json` contains no `tool_use`/`function`/`tool_call`. The "agent" reads a CONTEXT+QUESTION and emits an answer tag — it is **not** a tool-using, multi-step agent.
- **Real metrics from this run** (2 SQuAD val items): `squad-val-12` EM=1.0/F1=1.0 ("Levi's Stadium"); `squad-val-13` EM=0.0/F1=0.8 (predicted "Santa Clara, California" vs gold "Santa Clara" — a sub-string match, not exact). Scoring is exact-match / token-F1 on the answer tag.
- **Backend & tokens.** `summary.json` records `model_backend: claude_chat` (the `claude` CLI) with `claude-sonnet-4-6` as both optimizer and target. Token usage is **not** recorded — the run tracks EM/F1/`n_turns`, not tokens.

## Data sources

Every value above is from a real run — no fabricated rows (Constitution I):

- **Optimizer**: `skillopt==0.1.0` (`microsoft/SkillOpt`, cloned to `~/Documents/GitHub/SkillOpt-src`), `ReflACTTrainer` via `scripts/train.py`, `searchqa` env.
- **Eval data**: real **SQuAD** validation items (`rajpurkar/squad`) reshaped into SearchQA `{id, question, context, answers}` format (2 train / 4 val / 4 test) at `/tmp/abook_skillopt_split`.
- **Models**: target + optimizer = `claude-sonnet-4-6` via the `claude_chat` backend → `claude` CLI → AWS Bedrock (`us-east-1`, `AWS_BEARER_TOKEN_BEDROCK`). The stale `--thinking` CLI flag was dropped (`reasoning_effort=""`).
- **Run dir (on-disk state)**: `/tmp/abook_skillopt_run` — `summary.json`, `history.json`, `best_skill.md`, `skills/skill_v*.md`, and `predictions/<id>/conversation.json` (the trajectory log `reflect` parses). Real outcome: 1 step, 0 accepts / **1 reject** (gate rejected the edit), 11 calls / 1,439 tokens, ~39s.
- **Adapter**: `agentbook.adapters.skillopt_adapter.SkillOptOptimizer` (`sync_from_disk`, `load_results`, `load_trajectories`) — disk-mapping unit-tested against a durable copy of this run at `tests/fixtures/skillopt_run/`.
- **GEPA comparison column**: the live US1 run in `notebooks/gepa_demo.ipynb` (real Bedrock, `_gepa_run_07/` archives a larger prior run).